In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import math
import re
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile

import sys
sys.path.append('../..')
from mount_drive import mount_s_drive

In [2]:
mount_s_drive(subfolder='LCICM/Databases')

Password for ikarhul1: ········


Mounting S drive subfolder LCICM/Databases
The following files and folders have been mounted to /home/idies/workspace/SAFE/:
  mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0
  mimic-iv-1.0
  eICU
  mimic-iv-ecg-parquet
  Kevin
  cardiac_arrest_nantes
  ACCMPMAP
  archive.zip
  Kevin.zip
  mimic-iv-2.2.zip
  mimic-iv-ed-2.2
  Untitled.ipynb
  Simon
  mimic3wdb-matched
  Sampath
  MIMICIII
  MOVER
  MIMICIV
  ENGAGES
  AUMCDB
  mimic-iv-2.2


mkdir: cannot create directory '/home/idies/workspace/SAFE/': File exists


In [3]:
def nameSearchCardiacArrest(text):
    text = str(text).lower()
    patterns = [r'cardia.*rrest',
                r'cardio.*rrest',
                r'circulat.*rrest',
                r'\basystole',
                r'\basystolia',
                r'\bpea\b|pulseless elec.*act.*',
                r'post.*rrest']
    if any(re.search(pattern, text) for pattern in patterns):
        exclusion_patterns = [r'history|hx|h/o',
                              r'before cardiac arrest',
                              r'due to cardiac arrest',
                              r'neonatal',
                              r'newborn',
                              r'resp.*rrest',
                              r'during cardiac arrest',
                              r'during arrest']       
        if not any(re.search(pattern, text) for pattern in exclusion_patterns):
            return True   
    return False

def icdSearchCardiacArrest(text):
    text = str(text).lower()
    icd10_match = re.search(r'\bi46.*\b', text)
    icd9_match = re.search(r'\b4275\b', text)
    return bool(icd10_match or icd9_match)

In [4]:
def read_by_chunks(file_path,chunk_size=1e6,head='infer',compression=None):
    df_chunks = []
    num_chunks = 0
    
    for chunk in pd.read_csv(file_path,chunksize=chunk_size,header=head,compression=compression):
        df_chunks.append(chunk)
        if num_chunks % 20 == 0:
            print(f'Chunk {num_chunks} Processed.')
        num_chunks += 1
        del chunk
    print('Processing finished.')
    
    return pd.concat(df_chunks, ignore_index=True)

In [5]:
database_folder = '/home/idies/workspace/SAFE/'

In [6]:
dx_df = pd.read_csv(database_folder+'mimic-iv-2.2/hosp/d_icd_diagnoses.csv')
diagnoses_df = read_by_chunks(database_folder+'mimic-iv-2.2/hosp/diagnoses_icd.csv')
ed_diagnoses_df = read_by_chunks(database_folder+'mimic-iv-ed-2.2/mimic-iv-ed-2.2/ed/diagnosis.csv.gz', compression='gzip')
procedureevents_df = read_by_chunks(database_folder+'mimic-iv-2.2/icu/procedureevents.csv')
patients_df = pd.read_csv(database_folder+'mimic-iv-2.2/hosp/patients.csv')
admissions_df = pd.read_csv(database_folder+'mimic-iv-2.2/hosp/admissions.csv')
icustays_df = pd.read_csv(database_folder+'mimic-iv-2.2/icu/icustays.csv')
edstays_df = read_by_chunks(database_folder+'mimic-iv-ed-2.2/mimic-iv-ed-2.2/ed/edstays.csv.gz', compression='gzip')
ed_triage_df = read_by_chunks(database_folder+'mimic-iv-ed-2.2/mimic-iv-ed-2.2/ed/triage.csv.gz', compression='gzip')

Chunk 0 Processed.
Processing finished.
Chunk 0 Processed.
Processing finished.
Chunk 0 Processed.
Processing finished.
Chunk 0 Processed.
Processing finished.
Chunk 0 Processed.
Processing finished.


### Chief Complaint

In [7]:
ed_triage_df['name_search'] = ed_triage_df['chiefcomplaint'].transform(nameSearchCardiacArrest)

In [8]:
ca_triage_df = ed_triage_df[ed_triage_df['name_search']]
ca_triage_df = ca_triage_df.sort_values(['subject_id','stay_id'])
ca_triage_df = ca_triage_df.drop_duplicates('stay_id', keep='first').reset_index(drop=True)
duplicate_subjects = ca_triage_df.groupby('subject_id')['stay_id'].nunique()
duplicate_subjects = duplicate_subjects[duplicate_subjects > 1].index
ca_triage_df = ca_triage_df[~ca_triage_df['subject_id'].isin(duplicate_subjects)]

In [9]:
ed_ca_patients_df = patients_df[['subject_id','gender','anchor_age']].merge(ca_triage_df, on='subject_id', how='right')
ed_ca_patients_df = ed_ca_patients_df[ed_ca_patients_df['anchor_age']>=18]
ed_ca_patients_df = ed_ca_patients_df.merge(edstays_df[['subject_id','hadm_id','stay_id','intime','outtime','disposition']], on=['subject_id','stay_id'], how='left')
na_df = ed_ca_patients_df[ed_ca_patients_df['hadm_id'].isna()]
ed_ca_patients_df = ed_ca_patients_df.dropna(subset=['hadm_id'])

In [10]:
ed_ca_patients_df['hadm_id'].nunique()

421

### Diagnosis Observed in ED

#### Cardiac arrest encounters (non intraoperative)

In [11]:
dx_df['icd_code'] = dx_df['icd_code'].str.strip()
dx_df['long_title'] = dx_df['long_title'].str.strip()
dx_df['icd_search'] = dx_df['icd_code'].transform(icdSearchCardiacArrest)
dx_df['name_search'] = dx_df['long_title'].transform(nameSearchCardiacArrest)
ca_dx_df = dx_df[(dx_df['icd_search'])|(dx_df['name_search'])]

In [12]:
diagnoses_df['icd_code'] = diagnoses_df['icd_code'].str.strip()
ca_df = diagnoses_df.merge(ca_dx_df[['icd_code','long_title']], on='icd_code', how='inner')
ca_df = ca_df[~ca_df['long_title'].str.lower().str.contains('intraoperative')]
ca_patients_df = patients_df[['subject_id','gender','anchor_age']].merge(ca_df, on='subject_id', how='right')
ca_patients_df = ca_patients_df[ca_patients_df['anchor_age']>=18]
print('CA encounters:', ca_patients_df['hadm_id'].nunique())

CA encounters: 1973


#### With recorded ICU stay

In [13]:
ca_adm_df = ca_patients_df.merge(admissions_df[['subject_id','hadm_id','admittime','dischtime','admission_type','admission_location','edregtime','edouttime','hospital_expire_flag']],
                                 on=['subject_id','hadm_id'], how='left')
ca_adm_icu_df = ca_adm_df.merge(icustays_df, on=['subject_id','hadm_id'], how='inner')
print('With recorded ICU stay:', ca_adm_icu_df['hadm_id'].nunique())

With recorded ICU stay: 1728


#### Hospital admission is non elective

In [14]:
exclude_type = ['SURGICAL SAME DAY ADMISSION','ELECTIVE']
df1 = ca_adm_icu_df[~ca_adm_icu_df['admission_type'].isin(exclude_type)]
print('Hospital admission is non elective:', df1['hadm_id'].nunique())

Hospital admission is non elective: 1613


#### Patient goes through ED or straight to ICU

In [15]:
df2 = df1[(df1['hadm_id'].isin(edstays_df['hadm_id']))|(df1['admittime']==df1['intime'])]
print('Patient goes through ED (or straight to ICU):', df2['hadm_id'].nunique())

Patient goes through ED (or straight to ICU): 901


#### Diagnosis in ED

In [16]:
ed_ca_df = ed_diagnoses_df.merge(ca_dx_df[['icd_code']], on='icd_code', how='inner')
ed_ca_df = ed_ca_df.merge(edstays_df[['stay_id','hadm_id','intime','outtime','disposition']], on='stay_id', how='left')
df3 = df2[df2['hadm_id'].isin(ed_ca_df['hadm_id'])]
print('Diagnosis is observed in ED:', df3['hadm_id'].nunique())

Diagnosis is observed in ED: 326


#### Removal of duplicates

In [17]:
duplicate_subjects = df3.groupby('subject_id')['hadm_id'].nunique()
duplicate_subjects = duplicate_subjects[duplicate_subjects > 1].index
df4 = df3[~df3['subject_id'].isin(duplicate_subjects)]
df4 = df4.sort_values(['subject_id','hadm_id','intime'])
df4 = df4.drop_duplicates('subject_id', keep='first').reset_index(drop=True)
print('After removal of duplicates:', df4['hadm_id'].nunique())

After removal of duplicates: 326


### Combine

In [18]:
ids_df1 = ed_ca_patients_df[['subject_id','hadm_id']]
ids_df2 = df4[['subject_id','hadm_id']]
ids_df = pd.concat([ids_df1, ids_df2], ignore_index=True)
ids_df = ids_df.drop_duplicates()

In [19]:
len(ids_df)

531

In [20]:
filtered_diagnoses_df = ca_df[ca_df['hadm_id'].isin(ids_df['hadm_id'])]
filtered_diagnoses_df = filtered_diagnoses_df.sort_values(['subject_id','hadm_id','seq_num'])
filtered_diagnoses_df = filtered_diagnoses_df.drop_duplicates('hadm_id', keep='first').reset_index(drop=True)

In [21]:
ca_ed_df = patients_df[['subject_id','gender','anchor_age']]
ca_ed_df = ca_ed_df.merge(ids_df,on=['subject_id'],how='inner')
ca_ed_df = ca_ed_df.merge(filtered_diagnoses_df[['hadm_id','long_title']],on='hadm_id',how='left')
ca_ed_df = ca_ed_df.merge(admissions_df[['subject_id','hadm_id','hospital_expire_flag']],on=['subject_id','hadm_id'],how='left')
ca_ed_df = ca_ed_df.merge(icustays_df[['subject_id','hadm_id','stay_id','intime','outtime']],on=['subject_id','hadm_id'],how='left')
ca_ed_df = ca_ed_df.sort_values(['subject_id','hadm_id','intime'])
ca_ed_df = ca_ed_df.drop_duplicates('hadm_id', keep='first').reset_index(drop=True)

In [22]:
ca_ed_df.head()

,subject_id,gender,anchor_age,hadm_id,long_title,hospital_expire_flag,stay_id,intime,outtime
0,10004720,M,61,22081550.0,NaN,1,35009126.0,2186-11-12 19:55:00,2186-11-17 21:15:55
1,10010471,F,89,29842315.0,Cardiac arrest due to underlying cardiac condi...,1,32119961.0,2155-12-02 20:33:00,2155-12-07 18:19:18
2,10038688,M,46,25926997.0,NaN,1,33804579.0,2181-12-06 08:14:00,2181-12-22 20:27:01
3,10067389,M,76,23577021.0,Cardiac arrest due to other underlying condition,1,34081592.0,2116-02-10 23:55:00,2116-02-24 04:49:08
4,10079545,F,89,26201323.0,"Cardiac arrest, cause unspecified",1,35535213.0,2155-12-25 04:32:00,2155-12-25 23:18:26


In [23]:
ca_ed_df.to_csv('CA_ED.csv',index=False)